# Billing reconciliation (Nov 2026)
Compares the mainframe billing file with `dw.premium_billing` by product line.
Ad-hoc analysis by Data Platform; not scheduled.

In [1]:
import pandas as pd
stage = pd.read_sql('SELECT * FROM dw.premium_billing_stage WHERE load_id = 4182', conn)
dw = pd.read_sql('SELECT * FROM dw.v_premium_by_line', conn)
stage.merge(policyholders, on='policy_no').head(4)

      policy_no line_code   premium_amt  holder_name        date_of_birth  national_id  postcode
0  MTR000123456       MTR        451.20  Alys Penhallow     1984-03-17     AB123456C    ZZ1 4QT
1  HOM000998877       HOM        612.75  Tomasz Wiercinski  1979-11-02     CE654321A    ZZ9 7LP
2  PET000441122       PET        188.40  Imogen Treharne    1992-06-28     JK112233D    ZZ3 2RD
3  MTR000556677       MTR       1210.00  Dana Whitcombe     1985-09-14     123-45-6789  99999

In [2]:
diff = stage.groupby('line_code').premium_amt.sum() - dw.set_index('line_code').total_premium
diff[diff.abs() > 0.01]

line_code
GDG    -14.20
Name: premium_amt, dtype: float64

GDG differs by -14.20: one refund record (negative overpunch `}`) was dropped by the old loader. Fixed in loader 0.14.0.